In [22]:
import random

def LDPC_gallager(n, dv, dc):
    """Construction d'une matrice LDPC régulière par la méthode de Gallager."""
    # n : nombre de colonnes, dv : poids colonne, dc : poids ligne 
    if (n * dv) % dc != 0:
        raise ValueError("n * dv doit être divisible par dc")
    m = (n * dv) // dc
    
    rows_per_block = m // dv
    H_blocks = []
    
    # 1. Création du premier bloc de base
    block1 = matrix(GF(2), rows_per_block, n)
    for i in range(rows_per_block):
        for j in range(dc):
            block1[i, i*dc + j] = 1
            
    H_blocks.append(block1)
    
    # 2. Ajout des autres blocs par permutation de colonnes 
    for _ in range(dv - 1):
        perm = list(range(n))
        random.shuffle(perm)
        # On crée une nouvelle matrice en réordonnant les colonnes 
        permuted_block = block1.matrix_from_columns(perm)
        H_blocks.append(permuted_block)
        
    return block_matrix(dv, 1, H_blocks)

def LDPC_MacKay_Neal(n, dv, dc):
    """Construction d'une matrice LDPC régulière par la méthode MacKay-Neal."""
    m = (n * dv) // dc
    H = matrix(GF(2), m, n)
    
    for j in range(n):
        # On cherche des lignes qui n'ont pas encore dc "1"
        available_rows = [i for i in range(m) if H.row(i).hamming_weight() < dc]
        if len(available_rows) < dv:
            # En cas d'échec de contrainte, on recommence ou on ajuste
            return LDPC_MacKay_Neal(n, dv, dc) 
        
        chosen_rows = random.sample(available_rows, dv)
        for i in chosen_rows:
            H[i, j] = 1
    return H

In [23]:
def tanner_graph(H):
    """Génère le graphe de Tanner à partir de la matrice de parité H."""
    m, n = H.nrows(), H.ncols()
    G = Graph()
    
    var_nodes = ["v" + str(j) for j in range(n)]
    ctrl_nodes = ["c" + str(i) for i in range(m)]
    G.add_vertices(var_nodes + ctrl_nodes) # [cite: 594, 632]
    
    for i in range(m):
        for j in range(n):
            if H[i, j] == 1:
                G.add_edge(ctrl_nodes[i], var_nodes[j]) # [cite: 588, 595]
                
    return G

# Visualisation (Question 3.2)
def plot_tanner(H):
    m, n = H.nrows(), H.ncols()
    G = tanner_graph(H)
    var_nodes = ["v" + str(j) for j in range(n)]
    ctrl_nodes = ["c" + str(i) for i in range(m)]
    
    color_dict = {"red": var_nodes, "blue": ctrl_nodes} # [cite: 618-620]
    
    # Positionnement manuel pour alignement gauche/droite [cite: 624-629]
    pos = {}
    for j, v in enumerate(var_nodes): pos[v] = (0, -j)
    for i, c in enumerate(ctrl_nodes): pos[c] = (4, -i * n / m)
    
    return G.plot(vertex_colors=color_dict, pos=pos)

In [24]:
def bit_flipping(H, y, max_iter):
    """Algorithme de décodage Hard-Decision Bit-Flipping."""
    m, n = H.nrows(), H.ncols()
    y_curr = copy(y)
    
    for _ in range(max_iter):
        syndrome = H * y_curr
        if syndrome.is_zero(): # Succès [cite: 641]
            return y_curr
        
        # Compter le nombre de contrôles insatisfaits pour chaque bit
        unsatisfied_counts = []
        for j in range(n):
            count = 0
            for i in range(m):
                if H[i, j] == 1 and syndrome[i] == 1:
                    count += 1
            unsatisfied_counts.append(count)
            
        # Identifier le(s) bit(s) avec le maximum de contrôles insatisfaits
        max_unsatisfied = max(unsatisfied_counts)
        if max_unsatisfied == 0: break
        
        # Retourner (flipper) les bits critiques
        for j in range(n):
            if unsatisfied_counts[j] == max_unsatisfied:
                y_curr[j] = (y_curr[j] + 1) % 2
                
    return y_curr

In [26]:
import time
import random
import numpy as np

def test_bit_flipping_performance(n, dv, dc, p_range, nb_essais=20):
    """
    Évalue le taux de succès et le temps de calcul du Bit-Flipping.
    [cite: 601, 602]
    """
    print(f"Test LDPC (n={n}, dv={dv}, dc={dc})")
    print(f"{'p':<10} | {'Succès (%)':<12} | {'Temps Moyen (s)':<15}")
    print("-" * 45)
    
    try:
        # Suppression du [cite] qui causait l'erreur
        H = LDPC_gallager(n, dv, dc) 
    except ValueError:
        print("Erreur : Paramètres incompatibles.")
        return

    # Le mot de code nul est utilisé pour simplifier le test (Hx^T = 0)
    x_code = vector(GF(2), n) 

    for p in p_range:
        succes = 0
        total_time = 0
        
        for _ in range(nb_essais):
            # Simulation du canal BSC [cite: 596]
            erreur = vector(GF(2), [1 if random.random() < p else 0 for _ in range(n)])
            y_recu = x_code + erreur
            
            # Décodage Bit-Flipping [cite: 599]
            start = time.time()
            y_decod = bit_flipping(H, y_recu, max_iter=100)
            end = time.time()
            
            # Vérification 
            if (H * y_decod).is_zero() and y_decod == x_code:
                succes += 1
            total_time += (end - start)
            
        # Correction du formatage : conversion explicite en float pour les f-strings
        taux_succes = float((succes / nb_essais) * 100)
        temps_moyen = float(total_time / nb_essais)
        
        print(f"{float(p):<10.3f} | {taux_succes:<12.1f} | {temps_moyen:<15.5f}")

# Paramètres du TP
p_valeurs = [0.01, 0.03, 0.05, 0.07, 0.10]
test_bit_flipping_performance(600, 3, 6, p_valeurs)

Test LDPC (n=600, dv=3, dc=6)
p          | Succès (%)   | Temps Moyen (s)
---------------------------------------------
0.010      | 100.0        | 0.05302        
0.030      | 95.0         | 0.32592        
0.050      | 25.0         | 2.66147        
0.070      | 5.0          | 3.24529        
0.100      | 0.0          | 48.37164       
